# 06 · Explainability

Explain predictions and turn them into developer-facing advice (**RQ2**): global feature importance, SHAP & LIME (guarded), and a signals → mitigation-suggestions mapping.

- **Inputs:** `data/sample/sample_prs.csv`
- **Outputs:** Importance rankings, local explanations, and human-readable suggestions.

> ⚠️ **Sample vs. real data.** This notebook runs on the committed 10-row synthetic sample so the toolchain works without PRismBench. The sample has singleton classes, so metrics here are *illustrative only*. Each `TODO` marks where the real dataset in `data/raw/` plugs in.

In [ ]:
# --- Standard setup: locate project root, add src/ to path, load helpers ---
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    """Walk upwards until we find the repo root (has pyproject.toml + src/pr_risk)."""
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "src" / "pr_risk").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 50)
SAMPLE_CSV = PROJECT_ROOT / "data" / "sample" / "sample_prs.csv"
print("Project root :", PROJECT_ROOT)
print("Sample CSV   :", SAMPLE_CSV.name, "| exists:", SAMPLE_CSV.exists())

In [ ]:
import matplotlib.pyplot as plt

try:
    import seaborn as sns

    sns.set_theme(style="whitegrid")
    HAS_SNS = True
except ImportError:  # seaborn is optional; matplotlib is enough
    HAS_SNS = False
print("seaborn available:", HAS_SNS)

## 1. Train a quick model
Metadata features + Random Forest on `is_risky` (exposes `feature_importances_`).

In [ ]:
from pr_risk.data.load_data import load_csv
from pr_risk.features.metadata_features import create_metadata_features
from pr_risk.models.train_baseline import train_random_forest

df = load_csv(SAMPLE_CSV)
df = df[df["is_risky"].isin([0, 1])].copy()
X = create_metadata_features(df)
y = df["is_risky"].astype(int)
feature_names = list(X.columns)
model = train_random_forest(X, y)
print("trained RF on", X.shape[1], "metadata features")

## 2. Global feature importance

In [ ]:
from pr_risk.explainability.feature_importance import explain_with_feature_importance

importance = explain_with_feature_importance(model, feature_names, top_k=len(feature_names))
display(importance)
importance.set_index("feature")["importance"].sort_values().plot(
    kind="barh", figsize=(6, 3), title="Global feature importance (RF)"
)
plt.tight_layout()
plt.show()

## 3. SHAP (guarded)
Runs only if `shap` is installed; TreeSHAP is fast and deterministic for tree models.

In [ ]:
import numpy as np

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
print("shap available:", HAS_SHAP)

if HAS_SHAP:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    print("SHAP values computed:", np.asarray(shap_values).shape)
    # shap.summary_plot(shap_values, X, show=True)  # uncomment in an interactive session
else:
    print("Install shap to compute Shapley explanations (pip install shap).")

## 4. LIME (guarded)
Local explanation for a single PR instance.

In [ ]:
try:
    from lime.lime_tabular import LimeTabularExplainer
    HAS_LIME = True
except ImportError:
    HAS_LIME = False
print("lime available:", HAS_LIME)

if HAS_LIME:
    lime_explainer = LimeTabularExplainer(
        X.values, feature_names=feature_names,
        class_names=["non_risky", "risky"], discretize_continuous=True,
    )
    exp = lime_explainer.explain_instance(X.values[0], model.predict_proba, num_features=5)
    print("LIME explanation for row 0:")
    for feat, weight in exp.as_list():
        print(f"  {feat:30s} {weight:+.3f}")
else:
    print("Install lime to compute local explanations (pip install lime).")

## 5. Signals → mitigation suggestions
The explainability module converts model signals into practical advice
(`generate_human_readable_explanation`). This mapping is what the prototype surfaces to reviewers.

In [ ]:
from pr_risk.explainability.explanation_generator import generate_human_readable_explanation

SUGGESTIONS = {
    "lines_added": "Large diff — split the PR into smaller, focused changes.",
    "files_changed": "Many files touched — request additional reviewers.",
    "ci_failed": "CI failed — investigate failing checks before merge.",
    "reviewers_count": "Few reviewers — add a domain expert.",
    "commits_count": "Many commits — consider squashing / clearer history.",
    "comments_count": "High discussion — ensure concerns are resolved.",
    "lines_deleted": "Large deletions — confirm tests still cover behaviour.",
}

top_factors = importance["feature"].head(3).tolist()
suggestion_table = pd.DataFrame(
    [{"signal": f, "suggestion": SUGGESTIONS.get(f, "Review this signal.")} for f in top_factors]
)
display(suggestion_table)
print(generate_human_readable_explanation(1, "security_risk", top_factors))

## Next steps / TODO (real data)
- Implement `pr_risk.explainability.shap_explainer` / `lime_explainer` (currently placeholders).
- Use **LIME/SHAP agreement on top-k features as a robustness signal** (see `docs/README.md` §5).
- Run a VIF / correlation check on PR metrics; prefer group-level SHAP for correlated clusters.
- For transformers, add attention / integrated-gradients attributions.